In [ ]:
import numpy as np
from suite2p.io import BinaryFile
import matplotlib.pyplot as plt
from pathlib import Path
from cellpose import transforms
from tqdm import trange
from suite2p.extraction import preprocess
from cellpose import utils
from sklearn.decomposition import FastICA
from scipy.stats import zscore

mname = 'GT1'
root = Path('/home/carsen/suite2p_paper/GT1')
ipl = 1

def skewness(x):
    return x ** 3, (3 * x ** 2).mean(axis=-1)

db = np.load(root / f'suite2p/plane{ipl}/db.npy', allow_pickle=True).item()
settings = np.load(root / f'suite2p/plane{ipl}/settings.npy', allow_pickle=True).item()
F = np.load(root / f'suite2p/plane{ipl}/F.npy')
Fneu = np.load(root / f'suite2p/plane{ipl}/Fneu.npy')
stat = np.load(root / f'suite2p/plane{ipl}/stat.npy', allow_pickle=True)
npix = np.array([s['npix'] for s in stat])
ar = np.array([s['aspect_ratio'] for s in stat])
iscell = np.load(root / f'suite2p/plane{ipl}/iscell.npy')[:,1]
reg_outputs = np.load(root / f'suite2p/plane{ipl}/reg_outputs.npy', allow_pickle=True).item()
yrange, xrange = reg_outputs['yrange'], reg_outputs['xrange']
detect_outputs = np.load(root / f'suite2p/plane{ipl}/detect_outputs.npy', allow_pickle=True).item()
fs = settings['fs']
max_proj = np.zeros((db['Ly'], db['Lx']), 'float32')
max_proj[yrange[0]:yrange[1], xrange[0]:xrange[1]] = detect_outputs['max_proj']

Us = np.zeros((F.shape[0], 2), 'float32')
Vs = np.zeros((F.shape), 'float32')
for i in trange(F.shape[0]):
    X = np.stack((F[i], Fneu[i]), axis=1)
    X -= X.mean(axis=0)
    X /= X[:,0].std()

    ica = FastICA(n_components=2, fun=skewness, random_state=0, w_init=np.eye(2)).fit(X)

    U = ica.components_

    ica = ica.transform(X)
    V = (U @ (X - X.mean(axis=0)).T).T

    cc0 = (zscore(V, axis=0).T @ zscore(X[:,:], axis=0)) / V.shape[0]
    ic = np.abs(cc0[:,1]).argmax()
    ic = 1 - ic
    csign = np.sign(cc0[ic,0])

    Us[i] = U[ic] * csign
    Vs[i] = V[:, ic] * csign

cc = -Us[:,1] / Us[:,0]

In [ ]:
import importlib
import figures

fig = figures.suppfig_neuropil(db, stat, max_proj, F, Fneu, cc, iscell, ar, npix)

fig.savefig('figures/suppfig_neuropil.pdf', dpi=150)